<a href="https://colab.research.google.com/github/pavansaiv22/CSA1734-AI/blob/main/exp1%20with%20all(%201.1...).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Ex. No. 1.1   8-Puzzle Problem

In [ ]:
import heapq

GOAL = (1, 2, 3, 4, 5, 6, 7, 8, 0)

def manhattan(state):
    dist = 0
    for i, tile in enumerate(state):
        if tile:
            g = GOAL.index(tile)
            dist += abs(i // 3 - g // 3) + abs(i % 3 - g % 3)
    return dist

def neighbours(state):
    b = state.index(0)
    r, c = divmod(b, 3)
    for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):
        nr, nc = r + dr, c + dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            n = nr * 3 + nc
            s = list(state)
            s[b], s[n] = s[n], s[b]
            yield tuple(s)

def solve(start):
    pq = [(manhattan(start), 0, start, [start])]
    seen = set()
    while pq:
        f, g, state, path = heapq.heappop(pq)
        if state == GOAL:
            return path
        if state in seen:
            continue
        seen.add(state)
        for nxt in neighbours(state):
            if nxt not in seen:
                heapq.heappush(pq, (g + 1 + manhattan(nxt), g + 1, nxt, path + [nxt]))
    return None

def show(state):
    for i in range(0, 9, 3):
        print(' '.join(str(x) if x else '_' for x in state[i:i + 3]))
    print()

start = (1, 2, 3, 4, 0, 6, 7, 5, 8)
path = solve(start)
print("Solved in", len(path) - 1, "moves\n")
for step, s in enumerate(path):
    print("Step", step)
    show(s)

Solved in 2 moves

Step 0
1 2 3
4 _ 6
7 5 8

Step 1
1 2 3
4 5 6
7 _ 8

Step 2
1 2 3
4 5 6
7 8 _



Ex. No. 1.2   8-Queen Problem

In [ ]:
N = 8
solutions = []

def safe(cols, row):
    c = len(cols)
    return all(r != row and abs(r - row) != c - i for i, r in enumerate(cols))

def place(cols):
    if len(cols) == N:
        solutions.append(cols[:])
        return
    for row in range(N):
        if safe(cols, row):
            cols.append(row)
            place(cols)
            cols.pop()

place([])
print("Total solutions:", len(solutions))
print("First solution (row index for each column):", solutions[0])
for r in range(N):
    print(' '.join('Q' if solutions[0][c] == r else '.' for c in range(N)))

Ex. No. 1.3   Water Jug Problem

In [ ]:
from collections import deque

CAP_A, CAP_B, TARGET = 4, 3, 2

def successors(state):
    a, b = state
    pour_ab = min(a, CAP_B - b)
    pour_ba = min(b, CAP_A - a)
    return {
        (CAP_A, b): "Fill 4L jug",
        (a, CAP_B): "Fill 3L jug",
        (0, b): "Empty 4L jug",
        (a, 0): "Empty 3L jug",
        (a - pour_ab, b + pour_ab): "Pour 4L -> 3L",
        (a + pour_ba, b - pour_ba): "Pour 3L -> 4L",
    }

def solve():
    start = (0, 0)
    queue = deque([start])
    parent = {start: (None, "Start")}
    while queue:
        cur = queue.popleft()
        if cur[0] == TARGET:
            path = []
            while cur is not None:
                path.append((cur, parent[cur][1]))
                cur = parent[cur][0]
            return path[::-1]
        for nxt, action in successors(cur).items():
            if nxt not in parent:
                parent[nxt] = (cur, action)
                queue.append(nxt)

for state, action in solve():
    print(state, "<-", action)

(0, 0) <- Start
(4, 0) <- Fill 4L jug
(1, 3) <- Pour 4L -> 3L
(1, 0) <- Empty 3L jug
(0, 1) <- Pour 4L -> 3L
(4, 1) <- Fill 4L jug
(2, 3) <- Pour 4L -> 3L


Ex. No. 1.4   Crypt-Arithmetic Problem

In [ ]:
from itertools import permutations

def word_value(word, mapping):
    return int(''.join(str(mapping[ch]) for ch in word))

def solve(w1, w2, result):
    letters = sorted(set(w1 + w2 + result))
    leading = {w1[0], w2[0], result[0]}
    for digits in permutations(range(10), len(letters)):
        mapping = dict(zip(letters, digits))
        if any(mapping[ch] == 0 for ch in leading):
            continue
        if word_value(w1, mapping) + word_value(w2, mapping) == word_value(result, mapping):
            return mapping
    return None

m = solve("SEND", "MORE", "MONEY")
print("Mapping:", m)
print("  ", word_value("SEND", m))
print("+ ", word_value("MORE", m))
print("= ", word_value("MONEY", m))

Mapping: {'D': 7, 'E': 5, 'M': 1, 'N': 6, 'O': 0, 'R': 8, 'S': 9, 'Y': 2}
   9567
+  1085
=  10652


Ex. No. 1.5   Missionaries and Cannibals Problem

In [ ]:
from collections import deque

LOADS = [(1, 0), (2, 0), (0, 1), (0, 2), (1, 1)]

def valid(m, c):
    if not (0 <= m <= 3 and 0 <= c <= 3):
        return False
    if m and m < c:
        return False
    rm, rc = 3 - m, 3 - c
    if rm and rm < rc:
        return False
    return True

def solve():
    start, goal = (3, 3, 1), (0, 0, 0)
    queue = deque([start])
    parent = {start: None}
    while queue:
        cur = queue.popleft()
        if cur == goal:
            path = []
            while cur:
                path.append(cur)
                cur = parent[cur]
            return path[::-1]
        m, c, b = cur
        for dm, dc in LOADS:
            nm, nc = (m - dm, c - dc) if b else (m + dm, c + dc)
            nxt = (nm, nc, 1 - b)
            if valid(nm, nc) and nxt not in parent:
                parent[nxt] = cur
                queue.append(nxt)

path = solve()
print("Steps:", len(path) - 1)
for s in path:
    print("Left bank: M=%d C=%d | Boat: %s | Right bank: M=%d C=%d" %
          (s[0], s[1], "left " if s[2] else "right", 3 - s[0], 3 - s[1]))

Steps: 11
Left bank: M=3 C=3 | Boat: left  | Right bank: M=0 C=0
Left bank: M=3 C=1 | Boat: right | Right bank: M=0 C=2
Left bank: M=3 C=2 | Boat: left  | Right bank: M=0 C=1
Left bank: M=3 C=0 | Boat: right | Right bank: M=0 C=3
Left bank: M=3 C=1 | Boat: left  | Right bank: M=0 C=2
Left bank: M=1 C=1 | Boat: right | Right bank: M=2 C=2
Left bank: M=2 C=2 | Boat: left  | Right bank: M=1 C=1
Left bank: M=0 C=2 | Boat: right | Right bank: M=3 C=1
Left bank: M=0 C=3 | Boat: left  | Right bank: M=3 C=0
Left bank: M=0 C=1 | Boat: right | Right bank: M=3 C=2
Left bank: M=1 C=1 | Boat: left  | Right bank: M=2 C=2
Left bank: M=0 C=0 | Boat: right | Right bank: M=3 C=3


Ex. No. 1.6   Vacuum Cleaner Problem

In [ ]:
def reflex_agent(location, status):
    if status == "Dirty":
        return "Suck"
    return "Right" if location == "A" else "Left"

def run(env, location, steps=6):
    print("Initial:", env, "agent at", location)
    for t in range(1, steps + 1):
        action = reflex_agent(location, env[location])
        if action == "Suck":
            env[location] = "Clean"
        elif action == "Right":
            location = "B"
        else:
            location = "A"
        print("Step %d: action=%-5s -> agent at %s, world=%s" % (t, action, location, env))

run({"A": "Dirty", "B": "Dirty"}, "A")

Initial: {'A': 'Dirty', 'B': 'Dirty'} agent at A
Step 1: action=Suck  -> agent at A, world={'A': 'Clean', 'B': 'Dirty'}
Step 2: action=Right -> agent at B, world={'A': 'Clean', 'B': 'Dirty'}
Step 3: action=Suck  -> agent at B, world={'A': 'Clean', 'B': 'Clean'}
Step 4: action=Left  -> agent at A, world={'A': 'Clean', 'B': 'Clean'}
Step 5: action=Right -> agent at B, world={'A': 'Clean', 'B': 'Clean'}
Step 6: action=Left  -> agent at A, world={'A': 'Clean', 'B': 'Clean'}


Ex. No. 1.7   Breadth First Search (BFS)

In [ ]:
from collections import deque

graph = {
    'A': ['B', 'C'],
    'B': ['D', 'E'],
    'C': ['F'],
    'D': [],
    'E': ['F'],
    'F': [],
}

def bfs(graph, start):
    visited = [start]
    queue = deque([start])
    order = []
    while queue:
        node = queue.popleft()
        order.append(node)
        for nb in graph[node]:
            if nb not in visited:
                visited.append(nb)
                queue.append(nb)
    return order

print("BFS traversal from A:", bfs(graph, 'A'))

BFS traversal from A: ['A', 'B', 'C', 'D', 'E', 'F']


Ex. No. 1.8   Depth First Search (DFS)

In [ ]:
graph = {
    'A': ['B', 'C'],
    'B': ['D', 'E'],
    'C': ['F'],
    'D': [],
    'E': ['F'],
    'F': [],
}

def dfs(graph, node, visited=None):
    if visited is None:
        visited = []
    visited.append(node)
    for nb in graph[node]:
        if nb not in visited:
            dfs(graph, nb, visited)
    return visited

print("DFS traversal from A:", dfs(graph, 'A'))

DFS traversal from A: ['A', 'B', 'D', 'E', 'F', 'C']


Ex. No. 1.9   Travelling Salesman Problem

In [ ]:
from itertools import permutations

cities = ['A', 'B', 'C', 'D']
dist = [
    [0, 10, 15, 20],
    [10, 0, 35, 25],
    [15, 35, 0, 30],
    [20, 25, 30, 0],
]

def tour_cost(order):
    cost = 0
    for i in range(len(order)):
        cost += dist[order[i]][order[(i + 1) % len(order)]]
    return cost

best_cost, best_tour = float('inf'), None
for perm in permutations(range(1, len(cities))):
    tour = (0,) + perm
    c = tour_cost(tour)
    print(' -> '.join(cities[i] for i in tour + (0,)), "cost =", c)
    if c < best_cost:
        best_cost, best_tour = c, tour

print("\nBest tour:", ' -> '.join(cities[i] for i in best_tour + (0,)))
print("Minimum cost:", best_cost)

A -> B -> C -> D -> A cost = 95
A -> B -> D -> C -> A cost = 80
A -> C -> B -> D -> A cost = 95
A -> C -> D -> B -> A cost = 80
A -> D -> B -> C -> A cost = 95
A -> D -> C -> B -> A cost = 95

Best tour: A -> B -> D -> C -> A
Minimum cost: 80


Ex. No. 1.10   A* Algorithm

In [ ]:
import heapq

graph = {
    'S': {'A': 1, 'B': 4},
    'A': {'B': 2, 'C': 5, 'D': 12},
    'B': {'C': 2},
    'C': {'D': 3, 'G': 7},
    'D': {'G': 2},
    'G': {},
}
h = {'S': 7, 'A': 6, 'B': 4, 'C': 4, 'D': 2, 'G': 0}

def a_star(start, goal):
    open_list = [(h[start], 0, start, [start])]
    best_g = {start: 0}
    while open_list:
        f, g, node, path = heapq.heappop(open_list)
        if node == goal:
            return path, g
        for nb, cost in graph[node].items():
            ng = g + cost
            if ng < best_g.get(nb, float('inf')):
                best_g[nb] = ng
                heapq.heappush(open_list, (ng + h[nb], ng, nb, path + [nb]))
    return None, float('inf')

path, cost = a_star('S', 'G')
print("Path found:", ' -> '.join(path))
print("Total cost:", cost)

Path found: S -> A -> B -> C -> D -> G
Total cost: 10
